# Tech Challenge: Análise Estratégica do E-commerce Olist
## Uma Abordagem Baseada em Dados para Tomada de Decisão Executiva



### 1. Introdução
No comércio eletrônico, a eficiência das entregas e a satisfação do cliente são pontos críticos para o sucesso de uma empresa. A Olist atua facilitando esse processo, permitindo que lojistas menores consigam vender seus produtos em grandes plataformas de marketplace. Como essa rede envolve muita gente (clientes, vendedores, diferentes formas de pagamento e logística), o volume de dados gerado é enorme. Neste projeto, vamos explorar esses dados para extrair insights sobre o comportamento das vendas e das entregas.



### 2. Objetivo do Projeto
O objetivo deste projeto é explorar os dados brutos da Olist para entender a fundo a saúde financeira da empresa, a eficiência da logística e como o tempo de entrega afeta a satisfação do cliente. Para guiar o nosso desenvolvimento, estruturamos a análise em três perguntas principais:

1. Como está o ritmo de crescimento das vendas e o comportamento de compra? Vamos analisar isso olhando para o faturamento bruto (GMV), o volume de pedidos e o ticket médio ao longo do tempo.

2. A nossa operação de entregas é confiável e cumpre os prazos? Vamos investigar esse ponto calculando a taxa de SLA (cumprimento do prazo prometido) e o tempo real de entrega (Lead Time).

3. Qual é o impacto real da logística na nota do cliente? Cruzaremos os tempos de entrega com o Review Score (as notas de 1 a 5) para ver o quanto um atraso prejudica a avaliação da empresa.

A ideia final é levar esses dados de forma visual e interativa para o Tableau. Mas, antes disso, precisamos fazer todo o trabalho de preparação aqui no Python: limpar as duplicidades, tratar os nulos e juntar as 9 planilhas originais em uma base única e organizada para a análise.

## 3. Preparação do Ambiente e Importação das Bibliotecas

Para começar a mexer nos dados, o primeiro passo é importar a biblioteca Pandas, que vamos usar para carregar, limpar e manipular todas as tabelas. Logo em seguida, fazemos a leitura de cada um dos arquivos CSV que a Olist disponibilizou. Essa etapa é necessária para colocar todas as tabelas na memória do Python e podermos trabalhar com elas daqui para frente.

In [13]:
import pandas as pd

# Carregando as bases de dados originais que foram subidas no ambiente do Colab
orders = pd.read_csv('olist_orders_dataset.csv')
items = pd.read_csv('olist_order_items_dataset.csv')
payments = pd.read_csv('olist_order_payments_dataset.csv')
reviews = pd.read_csv('olist_order_reviews_dataset.csv')
customers = pd.read_csv('olist_customers_dataset.csv')
products = pd.read_csv('olist_products_dataset.csv')
translation = pd.read_csv('product_category_name_translation.csv')

print("Sucesso: Todas as tabelas foram carregadas e estão prontas para o tratamento!")

Sucesso: Todas as tabelas foram carregadas e estão prontas para o tratamento!


## 4. Tratamento de Granularidade e Resolução de Duplicidades

Durante a análise, percebemos um detalhe bem importante sobre a estrutura dos dados: a diferença na quantidade de linhas entre as tabelas. A tabela de pedidos (orders) tem exatamente uma linha para cada compra. Só que a tabela de pagamentos (payments) tem mais linhas do que pedidos.

Isso acontece porque, no e-commerce, é super comum o cliente dividir o pagamento de uma única compra — como pagar uma parte com um vale-presente e o restante no cartão, ou usar dois cartões de crédito diferentes. Se a gente juntar essas tabelas do jeito que estão, o Python vai duplicar as linhas dos pedidos que tiveram mais de uma forma de pagamento. O resultado disso seria péssimo: o nosso faturamento total calculado ia parecer muito maior do que é na realidade.

Para corrigir isso e não distorcer os nossos gráficos, vamos ajustar os dados usando o .groupby() antes de fazer qualquer junção:

Na tabela de pagamentos (payments): Vamos agrupar pelo código do pedido (order_id) e somar os valores (payment_value). Assim, cada pedido volta a ter só uma linha com o total gasto.

Na tabela de avaliações (reviews): Vamos agrupar por pedido e pegar a menor nota (review_score). Escolhemos fazer isso porque, se o cliente deu uma nota baixa para qualquer produto daquela compra, queremos registrar essa insatisfação para entender o que deu errado na entrega depois.

In [4]:
# Verificando as dimensões exatas da tabela de pedidos (Linhas, Colunas)
print(f"Dimensões do dataset de Pedidos (orders): {orders.shape}\n")

# Exibindo as primeiras 5 linhas para entender visualmente o conteúdo das colunas
print("Amostra dos dados de Pedidos:")
display(orders.head())

# Diagnosticando valores nulos e tipos de dados de cada coluna
print("\nDiagnóstico de estrutura e valores faltantes:")
orders.info()

Dimensões do dataset de Pedidos (orders): (99441, 8)

Amostra dos dados de Pedidos:


,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18 00:00:00
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13 00:00:00
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04 00:00:00
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15 00:00:00
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26 00:00:00



Diagnóstico de estrutura e valores faltantes:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 99441 entries, 0 to 99440
Data columns (total 8 columns):
 #   Column                         Non-Null Count  Dtype 
---  ------                         --------------  ----- 
 0   order_id                       99441 non-null  object
 1   customer_id                    99441 non-null  object
 2   order_status                   99441 non-null  object
 3   order_purchase_timestamp       99441 non-null  object
 4   order_approved_at              99281 non-null  object
 5   order_delivered_carrier_date   97658 non-null  object
 6   order_delivered_customer_date  96476 non-null  object
 7   order_estimated_delivery_date  99441 non-null  object
dtypes: object(8)
memory usage: 6.1+ MB


### 5. O que descobrimos olhando a tabela de Pedidos

Ao dar uma primeira olhada na tabela orders, já deu para notar alguns pontos bem importantes para o decorrer do trabalho:

A base tem quase 100 mil pedidos únicos (99.441 linhas), o que significa que temos bastante dado para analisar e tirar conclusões confiáveis.

Algumas colunas de data, como a order_delivered_customer_date (data em que o cliente recebeu o produto), têm valores nulos. Isso faz todo sentido com a lógica do negócio: se um pedido foi cancelado ou ainda está a caminho, ele não vai ter uma data de entrega mesmo. Vamos precisar tratar isso mais para a frente.

O Python leu todas as datas como texto (object). Então, já sabemos que vamos precisar converter essas colunas para o formato de data real (datetime) antes de calcular os prazos de entrega.

Agora que entendemos a dinâmica dos pedidos, vamos aplicar essa mesma lógica para investigar a tabela de pagamentos (payments).

In [5]:
# Verificando o formato dos dados de pagamentos
print(f"Dimensões do dataset de Pagamentos (payments): {payments.shape}\n")

# Entendendo o comportamento das transações com uma amostra
print("Amostra dos dados de Pagamentos:")
display(payments.head())

# Olhando estatísticas descritivas básicas dos valores pagos (Média, Mínimo, Máximo)
print("\nResumo estatístico dos valores de pagamento:")
display(payments['payment_value'].describe())

Dimensões do dataset de Pagamentos (payments): (103886, 5)

Amostra dos dados de Pagamentos:


,order_id,payment_sequential,payment_type,payment_installments,payment_value
0,b81ef226f3fe1789b1e8b2acac839d17,1,credit_card,8,99.33
1,a9810da82917af2d9aefd1278f1dcfa0,1,credit_card,1,24.39
2,25e8ea4e93396b6fa0d3dd708e76c1bd,1,credit_card,1,65.71
3,ba78997921bbcdc1373bb41e913ab953,1,credit_card,8,107.78
4,42fdf880ba16b47b59251dd489d4441a,1,credit_card,2,128.45



Resumo estatístico dos valores de pagamento:


,payment_value
count,103886.000000
mean,154.100380
std,217.494064
min,0.000000
25%,56.790000
50%,100.000000
75%,171.837500
max,13664.080000


###5.1. Análise Rápida de Pagamentos e Partida para a Ação
Olhando o resumo estatístico acima, deu para ver que os valores de pagamento variam bastante (tem desde valores bem baixos até compras bem altas), o que é normal para um grande e-commerce.

Agora que já inspecionamos os dados e confirmamos visualmente que os pedidos realmente se repetem quando há mais de uma forma de pagamento, chegou a hora de aplicar a estratégia que discutimos lá atrás: vamos rodar o .groupby() para unificar as tabelas payments e reviews, deixando tudo pronto e limpo no nível de um registro por pedido.

In [6]:
# Agrupando pagamentos por pedido para consolidar o faturamento real por transação
payments_agg = payments.groupby('order_id').agg({
    'payment_value': 'sum',
    'payment_type': 'first' # Mantém o primeiro tipo de pagamento como referência
}).reset_index()

# Agrupando avaliações por pedido para capturar o nível crítico de satisfação
reviews_agg = reviews.groupby('order_id').agg({
    'review_score': 'min'
}).reset_index()

print("Sucesso: Agregações concluídas! Dados normalizados para o nível de um registro por pedido.")

Sucesso: Agregações concluídas! Dados normalizados para o nível de um registro por pedido.


Como o nosso foco principal para o Tableau vai ser o valor total financeiro (payment_value), usamos o first para o tipo de pagamento apenas para não perder a coluna, sabendo que o valor somado é o que realmente importa para os nossos KPIs de faturamento.

## 5. Consolidação das Tabelas: Construindo a Base Unificada

Com os problemas de duplicidade completamente resolvidos e os dados normalizados, estamos prontos para realizar a grande união do ecossistema Olist.

Utilizaremos a técnica de relacionamento conhecida como *Left Join* (por meio do método `merge` do Pandas). Nossa espinha dorsal será a tabela de pedidos (`orders`) e, a partir dela, traremos sequencialmente as informações de itens, pagamentos corrigidos, notas de satisfação, dados demográficos dos clientes e as características dos produtos com suas respectivas traduções de categoria.

Ao final, transformaremos 9 arquivos fragmentados em uma única e poderosa matriz de dados.

In [7]:
# Unificando o ecossistema de dados a partir da tabela central de pedidos
df_master = orders.merge(items, on='order_id', how='left')
df_master = df_master.merge(payments_agg, on='order_id', how='left')
df_master = df_master.merge(reviews_agg, on='order_id', how='left')
df_master = df_master.merge(customers, on='customer_id', how='left')
df_master = df_master.merge(products,  on='product_id', how='left')
df_master = df_master.merge(translation, on='product_category_name', how='left')

print(f"Sucesso: Tabelas consolidadas! A nova base unificada possui as seguintes dimensões: {df_master.shape}")

Sucesso: Tabelas consolidadas! A nova base unificada possui as seguintes dimensões: (113425, 30)


## 6. Engenharia de Recursos (Feature Engineering) e Métricas de Negócio

Com as duplicidades resolvidas e os dados de pagamentos e avaliações organizados, o próximo passo é juntar tudo.

Para fazer isso, vamos usar a técnica de Left Join (através do comando pd.merge() do Pandas). A nossa tabela principal (a base de tudo) será a de pedidos (orders). A partir dela, vamos trazendo as outras informações aos poucos: os itens comprados, os pagamentos somados, as notas de satisfação, a localização dos clientes e os nomes das categorias dos produtos.

Usamos o Left Join justamente para garantir que nenhum pedido da tabela principal seja perdido no caminho, mesmo que falte alguma informação em outra tabela. No final desse processo, teremos todas as informações que precisamos concentradas em um único DataFrame para exportar para o Tableau.

In [10]:
# Convertendo as colunas de texto para o formato nativo de data do Pandas (datetime)
df_master['order_purchase_timestamp'] = pd.to_datetime(df_master['order_purchase_timestamp'])
df_master['order_delivered_customer_date'] = pd.to_datetime(df_master['order_delivered_customer_date'])
df_master['order_estimated_delivery_date'] = pd.to_datetime(df_master['order_estimated_delivery_date'])

# Calculando o tempo de entrega real em dias
df_master['dias_entrega_real'] = (df_master['order_delivered_customer_date'] - df_master['order_purchase_timestamp']).dt.days

# Criando o indicador de SLA (1 para entregas pontuais, 0 para atrasadas)
df_master['no_prazo'] = (df_master['order_delivered_customer_date'] <= df_master['order_estimated_delivery_date']).astype(int)

print("Sucesso: Transformação de datas concluída e novos KPIs logísticos integrados perfeitamente!")

Sucesso: Transformação de datas concluída e novos KPIs logísticos integrados perfeitamente!


## 7. Validação Final e Exportação para Business Intelligence (Tableau)

Para encerrar essa etapa do Python com tudo certinho, vamos fazer uma última inspeção rápida na tabela final só para conferir se todas as colunas novas foram criadas e se o DataFrame ficou com a estrutura esperada.

Com tudo validado, vamos exportar essa base consolidada para um arquivo único chamado olist_tableau_ready.csv.

Fazer isso aqui no Python poupa muito trabalho depois, porque evita ter que ficar fazendo cruzamentos complexos e pesados direto dentro da ferramenta de BI. Quando a gente carregar esse arquivo único no Tableau Public, o programa vai rodar de forma muito mais rápida e leve, facilitando a criação dos nossos gráficos e dashboards dinâmicos.

In [12]:
# Mostrando uma pequena amostra das novas colunas de logística criadas para validação visual
print("Amostra dos novos KPIs de Logística calculados:")
display(df_master[['order_id', 'dias_entrega_real', 'no_prazo']].head())

# Exportando a matriz de dados final para o arquivo CSV
df_master.to_csv('olist_tableau_ready.csv', index=False)

print("\nSucesso absoluto! O arquivo 'olist_tableau_ready.csv' foi gerado e está pronto para o download no menu lateral.")

Amostra dos novos KPIs de Logística calculados:


,order_id,dias_entrega_real,no_prazo
0,e481f51cbdc54678b7cc49136f2d6af7,8.0,1
1,53cdb2fc8bc7dce0b6741e2150273451,13.0,1
2,47770eb9100c2d0c44946d9cf07ec65d,9.0,1
3,949d5b44dbf5de918fe9c16f97b45f8a,13.0,1
4,ad21c59c0840e6cb83a9ceb5573f8159,2.0,1



Sucesso absoluto! O arquivo 'olist_tableau_ready.csv' foi gerado e está pronto para o download no menu lateral.
